In [8]:
from typing import Annotated, List
from langgraph.graph.message import add_messages  # or your own merge function

class State(TypedDict):
    transcription: Annotated[list, add_messages]  # or Annotated[List[str], add_messages]
    description: str
    chapters: str
    final_output: str

In [9]:
MODEL_VIDEO_DESCR = "gpt-4.1"
MODEL_VIDEO_CHAPTERS = "gpt-4o"
SYS_MSG_CHAPTERS = """
You are a helpful content creator assistant that generates accurate chapter sections for youtube videos.
You will write a chapter section with timeline stamps for the major parts of the video. You should initiate 
the section with 
'''📚 Chapters:'''

and the format should be:
 <double:digit timestamp> - <brief description of this section>
"""
SYS_MSG_DESCR = """
You are a helpful content creator assistant that generates accurate youtube video descriptions 
containing witty insightful and proper content given a transcription of a video containing its timestamps.
You write a very small paragraph, to the point.
"""

In [10]:
from openai import OpenAI

client = OpenAI()

def generate_video_description(prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL_VIDEO_DESCR,
        messages=[{"role": "system", "content": SYS_MSG_DESCR},
                  {"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

def generate_video_chapters(prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL_VIDEO_CHAPTERS,
        messages=[{"role": "system", "content": SYS_MSG_CHAPTERS},
                  {"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

def concatenate_final_video_description(youtube_video_description: str, youtube_chapter_section: str) -> str:
    full_video_description = f"""
        {youtube_video_description}.

        {youtube_chapter_section}

        🔗 Links:

        - Subscribe!: https://www.youtube.com/channel/UCu8WF59Scx9f3H1N_FgZUwQ
        - Tiktok: https://www.tiktok.com/@enkrateialucca?lang=en
        - Twitter: https://twitter.com/LucasEnkrateia
        - LinkedIn: https://www.linkedin.com/in/lucas-soares-969044167/
        - AI project tracker from Ben's Bites: https://bensbites.beehiiv.com/subscribe?ref=CoXhc4I0c0

        Support the Channel!

        - Buy me a cup of coffee: https://tr.ee/7tYsD-tUu2
        - Paypal: https://paypal.me/lucasenkrateia?country.x=PT&locale.x=pt_PT
        """
        
    return full_video_description

def write_video_description_to_file(video_description: str, filename: str = "youtube_description.txt") -> None:
    """
    Write the generated video description to a file.
    
    Args:
        video_description (str): The complete video description to write
        filename (str): The name of the file to write to (default: youtube_description.txt)
    """
    try:
        with open(filename, 'w', encoding='utf-8') as file:
            file.write(video_description)
        print(f"Video description saved to {filename}")
    except Exception as e:
        print(f"Error writing to file: {e}")

In [11]:
def transcription_input_node(state: State) -> State:
    return {**state,"transcription": state["transcription"]}
    
def generate_description_node(state: State) -> State:
    # Use your generate_video_description function here
    description = generate_video_description(state["transcription"])
    return {**state, "description": description}

def generate_chapters_node(state: State) -> State:
    # Use your generate_video_chapters function here
    chapters = generate_video_chapters(state["transcription"])
    return {**state, "chapters": chapters}

def concatenate_node(state: State) -> State:
    final_output = concatenate_final_video_description(state["description"], state["chapters"])
    return {**state, "final_output": final_output}

def write_to_file_node(state: State) -> State:
    write_video_description_to_file(state["final_output"])
    return state

In [12]:
from langgraph.graph import StateGraph,START


graph_builder = StateGraph(State)

graph_builder.add_node("input_transcription", transcription_input_node)
graph_builder.add_node("generate_description", generate_description_node)
graph_builder.add_node("generate_chapters", generate_chapters_node)
graph_builder.add_node("concatenate", concatenate_node)
graph_builder.add_node("write_to_file", write_to_file_node)

# Set up the flow: description -> chapters -> concatenate -> write
graph_builder.add_edge(START, "input_transcription")
graph_builder.add_edge("input_transcription", "generate_description")
graph_builder.add_edge("input_transcription", "generate_chapters")
graph_builder.add_edge("generate_description", "concatenate")
graph_builder.add_edge("generate_chapters", "concatenate")
graph_builder.add_edge("concatenate", "write_to_file")

graph_builder.set_entry_point("generate_description")
graph = graph_builder.compile()

In [13]:
from IPython.display import Image
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [14]:
video_transcription = """
Introduction to Using AI in the Terminal
0:00
and today we're going to talk about a few ways in which I use AI in the terminal starting with running models on
0:06
my terminal for that my main tool of usage is Simon Willis's LLM CLI if you
0:11
don't know who Simon Willis is he's just an amazing builder who makes these incredibly useful tools a lot of them
0:17
for the terminal and he was also the co-founder co-creator of Django if I'm not mistaken he's just an amazing guy
0:23
he's just one of the best guys to follow right now if you want to keep up with AI tools and AI in general so his ln CLI is
0:30
pretty awesome uh so essentially it's a CLI utility for interacting with large
0:35
language models it's extremely simple to use you can use it with pip install or with brew install if you're on a Mac uh
0:42
so or even you can even use it with UV which is really cool so you just copy
0:47
this pip install you open up a terminal you run this and when you do once you've done that uh you can you have to set up
0:55
your openi keys so you can run lm key set openai right and that will prompt you to
Setting Up LLMCLI by Simon Willison
1:02
set up your openi API key and once you've done that you can do things like "Hi tell
1:09
me three jokes in three different languages English
1:17
uh Portuguese and Italian."
1:23
And there we go you can interact with large language models directly on your terminal which
1:30
is pretty incredible now uh besides that you can also set up different models you
1:38
can do there's a bunch of functionality behind this which I really like uh one of my favorite ways of using it is with
1:44
different models so for example I can set up minus M and then I can set up
1:49
whatever model I want there's obviously a set of models that are supported and you can check that out by going to the
1:55
documentation and you can check out here all the things that you have available with the CLI and what you can check in
Exploring Models and Configurations with LLMCLI
2:02
to see which models are uh available for you is LM models so you can say LM
2:08
models and you can see all the models that you have available with the CLI so as you can see I have a bunch of openi
2:15
models and entropic models like claude etc and you can see here at the bottom that the default model is GPT4 mini
2:22
because it's like fast and good enough for most of the things that you would do in a terminal uh I usually use either
2:29
the default or GPT40 or claw 3.7 sonnet with at least with this uh LM CLI and
2:36
you can also set up local models by using the by installing the plug-in
2:41
GP4all which you can find here on the documentation as well so that's the
2:47
first one that I really like so check that out now moving on the other tool
2:52
that I really like is Olma so Olma is essentially an amazing tool for running
2:57
local models and it works perfectly on the terminal h essentially what you do is you go to Olma.ai
Running Local Models with Olamma
3:04
orma.com and here you can download the software that you're going to need so that you can actually run on your
3:10
terminal once you've done that you can u go to the Olma
3:17
GitHub and here once you've installed Olma on the software you can run this
3:25
command for example run lma 3.2 or run whatever model you want and that will
3:31
run the model in a chat interface so if I run LMA 3.2 now I'm interacting with
3:36
the model lma 3.2 if you've never downloaded the LMA 3.2 model on your
3:42
terminal that's going to download that for you automatically now the question that you can ask is Lucas how do I know
3:49
if I can download a model to my machine like it might be a model that's too big for your machine so what you have to do
3:55
is you have to go to.ai you go to models and then for example I love using Gemma 3 the
Managing Model Memory and Performance
4:02
recently released open source model from Google so you can come in here under that model specify the number of
4:09
parameters you want and for that you're going to see how much of memory that model is going to occupy as you can see
4:14
here the model is going to occupy like around 3.3 gigs and that's fine for me i have like 120 gigs of RAM in my machine
4:21
and a neuro engine GPU like with 40 cores it's like a powerful Mac so I can
4:27
run that and essentially most of the times you're going to be able to run up to like the 27 billion parameter 32
4:33
billion parameters which are around from like 15 to 20 gigs of RAM uh they're
4:39
that's going to require from your machine in order to run and more than that then you're going to have to look
4:45
at uh inference providers like Grock and others like VLM or SG lang etc but once
4:53
you've done that you can run as I said run whatever mod you want like for example GMA 3 and you can run a command
Working with Local Databases Using Data Set CLI
5:00
like hey uh write a one paragraph
5:09
essay on why the terminal is the best
5:15
tool for productivity i don't know whatever you want and that will run that prompt that
5:21
command automatically and this is running locally on my machine which is so awesome right now next thing in line
5:28
here folks is there's a bunch of other tools that you can run on your terminal there's probably there might be even
5:33
better ones that the ones that I'm talking about here but I really do like these ones there's also LMA CPP and a
5:39
bunch of LM Studio has a CLI to there's a bunch of other options that you can take a look now the second thing I want
5:46
to talk about is working with databases uh and this is something I'm just putting here because of the amazing data
5:52
set CLI invented by the one and only Mr simon Wilson and I really like this tool
5:58
especially because uh when I'm because I work so much with LLM I can do stuff like this which is if
Accessing and Filtering Interaction Logs
6:07
you go to the data set CLI GitHub so data set CLI GitHub right and you come over here
6:16
and you go this link you go down you go down and you run brew install data set right you run this once you've run this
6:23
you can run commands like this and uh let me just show you there you go data
6:29
set lm logs path I'm actually going to save this because I want to show you later how cool this is so I want to have
6:37
it here all saved up yeah and what's cool about this
6:43
command is that it will set up it will open up your database of all your interactions that you've had with the
6:50
LLM CLI so this is getting that the path to the logs folder where the database of
6:58
all your chats are stored and you open that up with data set so what you can do
Utilizing Embeddings with LLMCLI
7:04
is you will open up this link on your local host and now here we go and now you can see for example all if I click
7:11
on logs you can say let's if you can run this SQL command and you can have all
7:16
your you can have access to all your interactions with uh the LLM CLI so for
7:22
example if I go here to responses uh there you go 10 ideas for
7:30
whatever for whatever and we can filter by model so I can say GP4 mini we can
7:37
apply and now these are all my current my recent interactions with that model
7:43
and you can do all sorts of cool stuff with this i mean essentially you can use it for taking all of this information of
7:49
all my previous interactions with the LLM CLI and you know I'm going to actually do a presentation recently on
7:55
like the different ways that I use LLM so this information is going to be absolutely key so it's pretty cool you
Creating Embeddings and Searching for Similarities
8:01
can use data set for all sorts of things but this is definitely one of my favorites now the next example that I really like is working with embeddings
8:08
now if you don't know what an embedding is an embedding is just a vectorzed representation of text it means a list of numbers that represents the semantics
8:15
that's inside of a text and you can work with those in the terminal to quickly
8:20
search through a bunch of documents and things like that so for that we're going to use Simon Willis's again uh the LLM
8:27
CLI because it supports embeddings and it works super well so what you can do is pretty simple you go to the
8:34
embeddings with the CLI section in the LLM CLI documentation and you're going
8:39
to see examples on how to create a simple exa embedding for a phrase and then for multiple documents etc so if we
8:45
just copy this command here we can go to the terminal and then we can create an embedding for this phrase using this
8:51
model called three small and what you're going to get is just a list of a bunch of numbers which represent the embedding
8:59
of this phrase now pretty cool that on itself is not useful but what we can do we can use this to essentially create an
Converting Notes to Embeddings and Database Queries
9:06
embedding for potentially a bunch of phrases and then when you want to search through those you can run a command that
9:12
easily search to those ranks them according to how close they are to your query and then gives you an output so
9:19
for example I can do I create I can create this embedding and then I'm embedding this phrase my happy hound so
9:27
I do that and now what I can do is I can run a command called LLM similar which comes
9:34
with the LLM CLI and what I can do is I can look up okay similar phrases to
9:40
hound which obviously is going to return this right because it's the only one in this particular embedding and there you
9:46
go we just created that so super cool and what's even cooler is that we can do
9:53
that with for example a bunch of notes in a folder so I can run a command which is called embed multi and in this case
Understanding Embedding Scores and Context
10:01
I'm creating this embedding of uh my notes which I'm calling my notes i'm
10:07
using this model gene embeddings and from the files are in this folder on my current folder called sample notes and
10:12
it's all the markdown files and then I'm storing that in the notes
10:18
db and what's cool is when we when we do this we uh create this database this
10:27
SQLite database that we can then query with lm similar so in this case I'm
10:34
going to be I'm going to be getting all the similar information from this my
10:40
notes embedding that's stored in the notes database and I want similarity to this query prompts about learning and so
10:47
on so if I run this this is what we're going to get we run this and there you go we get
10:53
a ranking of all the embeddings that are similar to that we get the score so you see there's an 85% score here and this
Generating Screenshots with Shot Scraper
11:00
score is I'm not 100% sure but it's probably just a cosign similarity between the query and the embedded notes
11:07
and we get the content of those notes that's similar to what we were asking so you can see that in the file
11:14
AITOprompt.MD we had the content that was the closest to prompts about learning which makes perfect sense right
11:20
so this is pretty cool definitely check out uh LM CLI with all its capabilities it's freaking amazing now working beyond
11:28
text folks we can I love working with screenshots and again the one and only
11:33
Simon Willis has an amazing tool about that called Shot Scraper which essentially allows you to work with uh
11:40
create a full screenshot from a web page by just running a simple terminal command so I can run shots scraper and
11:47
in this case I'm getting the screenshot of the GitHub pit repo
11:52
for shots scraper and when I run this it's pretty cool because it saves locally here to this file right and I
Piping Data in the Terminal for Enhanced Productivity
12:00
had already run it once so it saved another image and when I open that up you can see that I get a full screenshot
12:07
of that repo which is pretty freaking awesome because then you can work with this to do different things and later on
12:13
this presentation we're going to see how to work with images with the LM CLI and
12:18
in different different interesting ways so working with screenshots is awesome and finally we can get to piping now
12:25
piping is an essential part of working with AI in the terminal because you're going to be able to get input from one
12:30
place and then take it to another place and so on and so forth so for example what I can do is I can say create a
12:40
fake database of people with their names
12:46
ages and interests uh uh as a
12:53
single.txt file and then with this command I'm going to save that to uh uh
Managing and Cleaning Terminal Output with Aliases
13:00
sample db.txt all right so what's going to happen is we're
13:07
going we're saving the outputs of this call to the LLM to this file called
13:12
sampled.txt and now I can do cat sampled.xt and you can see that we
13:17
created this fake database here uh it's not perfect we we got a little bit of
13:23
like some extra stuff that usually we can trim it but I'm just going to copy this and uh I'm just going to run
13:30
quickly i'm going to open up cursor in my terminal here i'm going to go to that sample db.txt i'm going to remove this
13:38
we're going to see how to do this automatically later but now we're just going to do it by hand now we have this
13:45
sample DB file age interest etc everything into that in a txt file so
13:51
now I'm going to come back to the terminal and what we can do now is I can say cat the sample db.txt txt and I can
13:59
pipe that to an LLM right so I can say something like for example
Using Bash and Python Scripts in Coordination with LLMs
14:06
uh retrieve the oldest
14:11
person in this DB with all their
14:17
info and now there you go 56 interest
14:22
making writing collecting all right so let's see 50 49 56 yeah that's the
14:27
oldest person in this DB that's pretty cool and it got the information correctly so the point is to show how
14:33
you can c the outputs of a file and pipe that with the pipe uh command uh with
14:38
the pipe operator to the LLM CLI and then do some information and you can do
14:44
that in many different interesting ways and we're going to see a few of those in just a sec so it's pretty cool uh we can
14:51
do different things and often we can clean up i have a bunch of aliases here
14:58
like clean markdown mermaid python html ginga bash diff because sometimes for
Building Customized Aliases for LLM Interactions
15:03
example I can say uh what is a command to list five random files in current
15:14
directory and show their contents right and I can pipe to clean
15:21
bash which is a little bash oneliner that cleans up it's either a bash
15:26
oneliner or a python script i don't remember now but it cleans up the outputs of this llm getting only the
15:33
command part that's going to be in the output of this model so when I run this folks this is what we're going to get
15:44
uh yeah I was supposed to get just this but I guess there was no bash it should
15:49
come with bash in order for this to work perfect but that's all right we can we
15:55
can say something like u generate a markdown file with a three paragraph
Incorporating Piped Output into Advanced Scripts
16:05
essay on how the terminal is awesome right so we can do that and then we can
16:12
say clean markdown so hopefully the output that contains the ticks and the
16:18
markdown part are going to be cleaned up from this uh from this output so if we wait a sec we should get there we go and
16:25
we get it perfectly we just get the markdown in that output as you can see here this is just the markdown which is
16:31
pretty awesome so piping is awesome you can use in many powerful ways now
16:38
creating aliases i mentioned the idea of creating aliases before and it's super fun when you integrate with LLMs for
16:44
example I could say something like alias summarize is equal to uh llm minus
16:52
m gpt 40
16:59
and then I can say uh summarize this and uh I can write this so that it
Integration with Clipboard for Fast Workflow
17:08
takes in input within the terminal so when I do this and I'm going to double
17:13
quotes here and double quotes here so we don't get messed up we don't mess up i
17:19
think that should work summarize i'm going to say summarize this info there you go okay so
17:27
now that I have this command summarize this info what I can do is for example
17:32
if I go to my sample DB that we just created right and I say cat sample DB
17:39
and then I pipe this to summarize this info hopefully what
17:45
we're going to be able to get is a summary of this file with just a single
17:53
command there you go this is awesome this list contains a bunch of da and
17:58
there you go we get a summary of that information which is really really cool you even got more information that I
Capture and Manipulate Output from LLMs
18:03
wanted so you can do that with a bunch of things for example you could
18:09
uh you could do this to summarize things to create specified summaries with
18:14
bullet points you can create all sorts of fancy interesting templates and aliases are super powerful and then if
18:21
you want to reuse them forever in your terminal you go to uh you go to
18:27
your you go to your aliases file that usually would be on your root folder from your from your machine it's called
18:34
alaliases and then in there you would save this command that you just did alias
18:39
etc etc save that in another line and then you're good to go you can reuse this forever every time the terminal
18:46
runs it's going to load up those aliases and you're going to be able to reuse this command which is super cool awesome
18:51
amazing so alases are awesome folks now working with the clipboard this is one of my favorite things because uh PB copy
Chaining Prompts for Tiered Information Processing
19:00
and PB paste are really like time savers for me so we can combine this idea of piping files with quick copy and paste
19:09
in order to be you know agile when working with LLMs for example I could
19:15
come over to the website here i come over to for example this section of the embeddings with the CLI
19:21
um in the documentation here for Simon Willis's tool i can I can hit Ctrl A and
19:28
then Ctrl C so now I have this on my clipboard so what I did is I could run
19:35
PB paste and then pipe that to lm and I'm going to go
19:41
gbt40 and then I could say so now this model has access to the contents of my
19:47
clipboard through this command pbaste and then I could say uh extract from
19:53
this documentation all the relevant commands to work with
Exploiting PDF to Text for Document Insights
20:00
embeddings in the terminal properly indexed into a
20:08
markdown style and I'm going to get fancy i'm going to say clean
20:13
markdown and I'm going to save it as embeddings CLI
20:21
maincomands.md so what's happening is hopefully I'm going to paste from my clipboard i'm going to pipe that to a
20:27
model and run this prompt and then I'm going to clean it up with clean markdown and then I'm going to save that to a
20:34
file called embedding main clis etc so when I run this hopefully everything's going to work and we're going to be able
20:41
to see the results so one two three
20:51
and and
20:58
There you go so now if I go to embedding CLI there you go folks all the commands
Introduces git ingest and Reminders of Token Cost
21:04
from that embedded embedding CLI
21:13
embeddev all right i'm not trusting this very much embed delete that makes no
21:19
sense let me see embed delete nope
21:24
so I think we had a maybe an issue of me not copying properly the contents of the
21:32
the CLI so let me just try that again so I'm going to go control arr C
21:40
and now I'm going to go PB paste pb
21:45
paste all right so that's working beautifully now PB paste beautiful and
21:50
now we're going to do that yeah all right so now we run cat
21:56
embed now folks as you can see all the proper commands have been extracted from
Utilize r.jina.ai for Webpage to Markdown Conversion
22:02
that documentation and saved to this file in the style that we were going for which is pretty awesome you see embed
22:08
multi you see embed multi embed embedding the phrases everything got saved properly so this is pretty cool so
22:16
the thing is I don't like writing all of this so I wrote a little alias co where
22:22
pbaste for me is just the letter P and PB copy which copies to my clipboard is
22:27
the letter C so I can do uh for example we
22:34
can for example uh paste the outputs of this documentation and then say to the model
22:41
summarize these docs into two sentences and when I run this it should
22:49
take the output of my clipboard and summarize it quickly and you get that output beautiful store embeddings etc
22:56
everything's correct so this is just amazing it's fast it works perfectly and I just have to write the letter P to
Tools like Repo Mix and Files to Prompt for Efficient File Management
23:03
take whatever and paste it send that to an LLM and I can do that the I can do the same in order to get a conversation
23:10
or an explanation from an LLM and paste that to my clipboard so I can say ln minus m gppt40 and then I can uh send it
23:20
a prompt like write down 10 ways I can
23:25
use the terminal to increase my productivity something like that and
23:31
then I can do pipe C and that's going to be the same as writing pipe PB copy
23:38
which means it's going to copy to my clipboard so when I hit this nothing's going to be shown in the terminal
23:43
terminal but I'll show you in a sec that something was copied to my clipboard automatically so now if I open up a
23:50
terminal here I open up cursor and I just write see all of that answer from the model got saved to my clipboard so
23:57
it's pretty cool so automating repetitive tasks efficient file management command line tools blah blah blah everything got saved here so it's
Agentic Use Cases with Autogen and Magenta CLI
24:04
pretty cool so that's another way of working with um AI in the terminal that
24:09
I really like by integrating this idea so definitely check this out now uh
24:15
we've already talked about that we already showed this these examples we already showed those so one thing that
24:21
you can do with piping that's pretty cool is you can chain prompts together that's another interesting way of
24:26
working with AI so what you can do is you can do something like for example uh I can write LLM
24:34
uh write the this these 10 examples here
24:40
right and we can we can paste that to the clipboard and we can say
24:51
uh write uh we can say let's say write write three
24:57
jokes about AI and uh life whatever it is and
Claude's Generative Capabilities in Project Building
25:06
then we can pipe that and we can say give each of these jokes a title and
25:14
output the title plus the joke and as bullet point something like that and now
25:21
what we're doing is this is the first prompt that's writing three jokes about AI and life and then the second prompt
25:27
is giving these jokes a title and then getting the joke plus the title as bullet points all right and uh when I
25:35
run this we're going to send the outputs of this first prompt to this model which
25:40
is going to process this one and then we're going to get there you go we get all that information so we get like a
25:46
title and a joke a title and a joke etc and you can do that you can expand this to do whatever you want and you can do a
25:52
bunch you can do multiple prompts etc so it's a pretty interesting way of working with AI in that sense all right so
25:58
chaining prompts is really fun and you can obviously for example when it comes
Exploring Python Scripts with UV for Smooth Execution
26:03
to chaining prompts so folks I have a paper here on my folder right it's about using prompts to teach learners how to
26:10
effectively use AI code generators great so what we can do that's really interesting when uh thinking about this
26:16
idea of chaining prompts together is that we can run a tool called PDF to
26:23
text right and we can send that paper.pdf right so this is going
26:30
to get the text from this PDF so when now I have the paper as a text file
26:38
right here so what I can do is I can say catpaper.txt and I can pipe that to a
26:43
model i'm going to pipe that to gt40 and I'm going to say
26:48
um write a comprehensive list of all the
26:53
main findings and insights from this paper
26:59
right and then I can pipe that to and then I can save that to a file like uh
Utilizing Inline Metadata for Efficient Packaging
27:05
paper insights for example insights.txt right so you can work with different documents in interesting
27:11
different ways and this is one of my favorites to like the summaries of papers real quick so now if I go paper
27:18
insights we have all the insights of that paper rise elements in computer education prompt engineering pedagogy
27:25
etc so it's a pretty cool way now folks some fun interesting repositories that I think you should take a look that can
27:31
complement the usage of LLMs in the terminal there's a bunch but like I noted a few that I really like i'm going
27:38
to start with uh actually I'm not going to start with ripple mix i'm going to start with git ingest now folks if you
27:44
go to any GitHub repo so I'm going to go to lmcli github repo okay so this is the
27:52
here there you go uh that's not the GitHub repo wait where is the LM
27:58
CLI ammo Wilson where is that github repo there you go okay this is the repo
Demonstrating UV-Run Python Scripts Step-by-Step
28:04
so if you go to any repo and you go to the URL and you come over here and you write gitingest.com so you replace the
28:11
GitHub part with just git ingest what you're going to get is this which is an
28:17
amazing free tool that essentially takes your entire repo and converts it into a
28:22
single file that you can send to a large language model now there's so many different interesting ways that you can
28:28
uh use this for uh the simplest one would be to copy all the contents of that repo and then pipe that through
28:34
your clipboard the way that I just showed you before to get answers and outputs from an LLM this is a really fun
28:41
way however you have to take a look at the estimated tokens which is this tool also provides which is amazing because
28:47
for example 171.1,000 tokens is more than the context length of a model like
28:53
GP40 however it is not more than the context length of claw 3.7 sonnet for
28:59
example so I could say copy all and I could do something like this i could
Using Python for LLM Output Navigation and Automation
29:04
paste that and then pipe it to uh claude 3
29:12
uh 7 sonnet i think it's 3.7 sonnet if I'm not mistaken and then I could say
29:18
something like summarize this repo and give its
29:25
file and give it its core file structure all right so we're going to do that and
29:31
we're going to see what happens so we're going to wait because it's a lot of tokens to process so we're
29:38
are we are making calls to API calls so remember that the the cost associated with tokens is going to be involved and
29:47
there we go summarizing the right repo so it got all that information so it's created a beautiful summary it's getting
29:53
the core stuff and the core info this is pretty cool and there you go i mean git
29:58
ingest is extremely useful now on the same style of git ingest folks you also
Setting Up Claude Projects for Script Creation
30:03
have something called r.gina.ai which is from a company called Gina which is really cool and
30:09
essentially what we can do with this is if I go to for example the
30:15
uh web page whatever web page I want for example in this document uh this
30:20
documentation page from the lmcli we're using that a lot and I go to the beginning of the URL and I write uh
30:28
r.ja.ai slash this is going to transform this web page into a markdown file that
30:34
I can send to a model so now this is just a single file with all that
30:39
information in a markdown style which is much more appropriate for LLMs so I can run Ctrl ARL C and now I can do the same
30:48
thing that I just did before so this is another tool to work with LLM because you can essentially just copy all of
30:54
this and send that to whatever model you want you don't even have to send it in the terminal you can send it to CHPT
Using Jinja Templates for Customized Text Processing
31:00
whatever but it's another different interesting way of working with
31:05
now moving on this is another one that I really like moving on uh Repix does the
31:12
same thing as Gitingest but it's something you can install locally on your folder so RepoMix is really cool uh
31:19
essentially what Repix does is this if I go over here and I go repo
31:25
mix so this is a tool to essentially um
31:30
pack an entire repo into a single AI friendly file so you can do the same
31:36
thing that git ingest does but with ripple mix you're doing that in your own
31:41
uh in your own local uh computer so maybe I clone a repo if I clone this
31:47
repo I could use repo mix in my local machine and then get everything about
31:53
this repo into a single file so repo mix is another one that I really like and I I use it quite a bit now in the same
Final Demonstrations with Mermaid, OCR, and Image Processing
32:00
style and again the amazing Simon Willis has an really cool tool called files to
32:06
prompt which does the same type of thing but for any type of folder any type of
32:11
structure any type of whatever so it's really cool you just run pip install files to prompt you go to terminal and
32:17
then for example I have a bunch of files here right uh this is like really a bunch of files but uh I'm just going to
32:24
remove for example from my sample notes i'm going to remove all the files there because I just copied those from my
32:30
original notes there you go uh I'm going to remove this folder sample
32:36
notes simple notes yes and now I'm going to like make a file called images and
32:43
then I'm going to move all the PNG files to the images and then I'm going to move make a file called txt files and then
32:49
I'm going to move all the txt files to that folder and the same thing with MD
32:57
files move all the uh moved files to MD files beautiful so now and I'm going to
Exploring Layered Control for AI-Driven Terminals
33:05
do the same PDFs uh PDF files move PDF to PDF files beautiful so now I have
33:12
these folders what I can do is I can run files to prompt and what's going to happen is oh I have to give the the
33:20
right uh folder obviously there you go so what you get with files to prompt is this single output containing every
33:28
single every single thing that you have in that current folder and all the nested folders so if I go files to
33:35
prompt and if I write files to prompt I can copy to my clipboard obviously or I
33:42
can save it to single lm ready
33:47
file.txt and it will not do that for images so we're getting errors because
33:53
it tried to do that for images so that's the one thing you're going to get an error when you work with images but everything that's not an image got or uh
Closing Remarks and Future Considerations
34:00
sorry not an image or a PDF got saved to this.xt txt file got indicated there so
34:05
if I go cat single file we get all that information from all the contents of everything into a single file so it's
34:12
all that content into a single file so it's another tool that's really useful obviously you can write a single bash
34:17
command to do that but it's nice to have like a like something that does that for us automatically so and again it's by
34:23
the one and only the great Simon Willis come on how can you not love this guy
34:29
now uh the another thing I want to talk about is agentic use cases which I
34:35
really really like agentic use cases in the terminal essentially mean uh being able to execute stuff perform actions so
34:42
far we've been talking about creating files editing files we've we've been talking about this idea of okay send and
34:47
making API calls but how about taking action in your terminal and there's a bunch of interesting ways you can do
34:54
that uh now the LLM CLI doesn't necessarily uh support search but there
35:01
are some options with the um plugins like there's the open router there's
35:06
Gemini that I think can do search there's a few options there but you can
35:11
do you can do some fancy things with uh aentic use cases in the terminal for
35:16
example there's a bunch of examples of simple simple terminal agents like from autogen there's something called magentic 1 CLI and magentic 1 CLI is
35:24
actually just a very simple tool from uh the folks that created the new 0.4 four
35:29
version of autogen and you can just run pip install magetic 1 cli and if you do
35:34
that so I'm going to run that right now here there you go so it's going to work
35:41
with playright so it's going to do browser automation things like that and when you do that we can I'm just going
35:46
to copy this prompt find flights and instead of from Seattle to Paris I'm
35:52
going to say from Lisbon to Ireland and format the result in a table
35:58
so Let's see what happens so what's really cool about this too is that it's going to work in the terminal but it's going to do stuff outside the terminal
36:04
by using a browser and uh hopefully this little simple demo is going to work and
36:10
we're going to be able to see that in you know live working in just any moment
36:17
now there you go and as the agent works we're going to be able to see what the
36:22
agent is doing what is the information that's getting and all of that good stuff so to answer this request we have
36:28
assembled the team so it assembled a bunch of agents to do this thing for me computer computer
36:34
terminal give verify fax to lookup fax to review so now when it goes to a page
36:41
we can actually see it this is a really cool uh feature of this uh use case we
36:47
can see what it's doing we can see the pages that it's accessing so it's really nice to see it work and then get that
36:53
information and to have that like compact into like a single tool that you can just run in the terminal it's pretty
36:58
freaking awesome right so it's going through pages is browsing usually it's not like a perfect use case it might get
37:04
stuck in some page it might get stuck so you have to be aware of that and just test it out it's not something that I
37:11
would say you should use on a daily basis but it's something interesting to experiment with and I'm gonna stop the
37:18
execution of this agent but you should definitely you know go experiment for yourself with your specific use case so
37:23
another one that I really like is if we go here on is the LM term which is like
37:29
a simpler version of this essentially for you to run simple local commands in your terminal so lm term is a simple
37:37
tool that allows you to do commands like list files in the current directory and it will create that command and then
37:42
execute it for you it's it's something that you can implement kind of like in a simpler way by just doing uh like a a
37:50
little bit of hack tricks with bash and uh the llm CLI but it's another one that
37:56
I kind of I think it's kind interesting because the concept is running it creates a command in your terminal and
38:02
then runs it and you can you know say okay yes you can run that command don't run that command etc etc so it's pretty
38:09
cool now another example that I like is cloud code which is in research preview right now clot code is essentially a
38:15
tool by Enthropic the team that created Claw 3.7 Sonic which is one of the best models in the world for coding and they
38:22
have this tool that essentially allows you to create an entire project with code an entire app everything from the
38:28
terminal it's similar to another tool that I like called Ather which does a
38:34
similar thing ather actually came before but a lot of people are like super hyped about claw code but Ader and Claw do
38:40
essentially the same thing you can do uh development in your terminal for example
38:46
uh I'm going to just say claude uh aentic coding i'm going to create a
38:51
folder called agentic coding i'm going to go over there and if you've already installed claude with this command npm
38:57
install which requires you to have noode.js 18 plus installed in your machine run claude and then you can say
39:04
with your permission yes it can execute files and now you get this very nice UI
39:10
to set up whatever project you want like for example I could say
39:16
uh build me a simple UI in Python to download and
39:24
visualize cool images from free APIs like Naza Naza or there's another one
39:33
Unsplash I think and Cloud is going to work autonomously within your machine
39:39
and get stuff done for you it's pretty amazing it works really well uh working
39:44
with Python scripts folks now I could do a full video just on this topic and I'm not going to do that because it's way
39:49
too much work uh I'm actually going to just for now I could I'm just going to
39:54
show you like a few summary examples but I will probably do like a single video just on this topic because it's
40:00
fascinating and I have a full course actually about this in O'Reilly if you
40:06
want to check that out it's called automate tasks with uh let me see I
40:11
forgot the name now automate tasks yeah using AI tools in Python to automate tasks folks doing a little bit of
40:18
marketing for myself right now it's a really cool course if you want to check that out it's essentially I'm going to
40:23
show how to do the stuff that I'm going to summarize now and essentially what this allows you to do is you can explore
40:31
uh different ways of building simple Python scripts in a way using a tool called UV for package management that
40:39
you generate a single file that is a tool on itself and there's different interesting ways that you can integrate
40:45
the idea of MCP into this but I don't want to get too complicated right now so uh I'm going to show you just a few
40:50
examples all right as you can see here Claude is already working really well and it seems like it worked this is
40:56
pretty cool see if it worked image viewer that's pretty cool let's see let's see let's see and if I open it up
41:04
oh look at that that's amazing fetching an image that's so cool unsplash fetch an
41:11
image all right unsplash didn't work dog API all right let's fetch an image ah that's
41:17
amazing that's so cool folks i just made this with claw in like a like one single
41:23
prompt and with the basic requirements from my own uh virtual environment it it
41:29
already works perfectly it's just the coolest thing ever right so yeah that's the power of agentic use cases in the
41:36
terminal so check that out but now we're talking about uh Python scripts so for
41:41
that is actually pretty cool so if I I'm going to create a folder here pi scripts right and I have a few examples that uh
41:50
I generated in the past that I think are worth mentioning so one of them is called navigate the llm output so I'm
41:57
just going to copy that over here the other one is I think it's called change this llm.py which is I'm going to save
42:04
here and let me see what is the other one that I was going to show searching through Anki flashcards there you go so
42:10
if I go I think it's called find yeah
42:16
there you go so folks in these examples uh these tools essentially allow you to
42:23
I I created these scripts which we can take a look here on cursor uh let's go to let's
42:32
go to navigate navigate lm output and my idea with these tools was to uh complement
42:39
the use of AI in the terminal with the missing aspect of it so like there's
42:45
missing UI aspects of working with the AI in the terminal there's missing information fetching there's a there's a
42:52
bunch of things there are gaps right and you can write Python scripts to patch those gaps and what's really cool is in
43:00
uh in this example here actually we're I'm I'm going to just generate a full example from scratch and then show in
43:05
these examples uh how they kind of relate so let let's just come over here
43:11
on this thing and we can say I'm going to use cloud 3.7 sonic for that
43:18
uh and we're going to create some Python scripts and what we're going to do
43:24
that's different I'm actually going to demonstrate with cloud uh projects first and then do an example in the terminal
43:31
so if I open up claude.ai AI here on my browser and I go to my projects which
43:39
should be right here there you go i have a project called Python tools there you go and this project and
43:46
if you don't know about cloud projects is where you can just essentially create uh a project where Claude integrates
43:52
with a bunch of files and instructions to create really specialized files and reply to
43:59
information etc you can do that for studying you can do that for research you can do that for generating projects
44:04
following instructions and I have a set of instructions that if I go over here I have a set of instructions for creating
44:10
Python scripts with a very specific format and this format follows um
44:17
the this very interesting idea that I got from this article by uh Simon Willis
44:23
about leveraging UV to generate single file Python scripts that run out of the
44:31
box you don't have to worry about package management and virtual environments and so on it's really cool
44:36
so the inspiration for this obviously was again the one and only the great Simon Willis wrote a really fascinating
44:42
article called building Python tools with a oneshot prompt using UV run and cloud projects you might imagine why I'm
44:48
using cloud projects right i don't think I did as good of a job as he did obviously but it's pretty cool the idea
44:53
is if you if if you go to the UV documentation
45:01
uh single scripts I think it's called there you go running scripts there you
45:06
go there's this idea that if you write a Python file and you add some inline
45:13
metadata to the file like in this example here as you can see and it you
45:18
specify within the file the dependencies for running that file uh UV will parse that metadata and set
45:26
up everything for you automatically and run the file out of the box so this is so cool because for example if I just
45:32
copy this file here and I I'm going to paste that information to
45:40
claude i'm going to say uh leveraging this idea of UV inline metadata i'm
45:49
actually going to do even better i'm going to copy everything from this page and I'm going to do it in the way that I
45:54
showed you before so gina.ai so that I get all the info from
45:59
this documentation there you go there you go control A and now I'm
46:06
gonna say uh create a single Python file
46:12
that can be uh executed out of the box with UV
46:20
run to uh let's see uh
46:27
plot some interesting data from some
46:36
free easy to access API and show that in
46:41
a nice dashboard leveraging mapplib and seabour which seabour is a
46:50
bit tougher so like you'll see what happens and I've never tested this before so let's see simple plotpy it's
46:57
going to be the name and I want to clean this the python script
47:04
first so I'm going to say clean Python and then sample plot.py i don't know if it's going to
47:10
work perfectly but let's take a look so I'm pasting that information from the UV documentation i'm sending that to Claude
47:15
i'm asking for the script i'm hopefully cleaning up the Python tick and then I'm saving that to a file called Python
47:22
sample plot.py so let's see if that works well command not found sample whatever okay uh but we have the sample
47:30
plot which if I just do cat okay that looks like it worked it looks like it
47:36
worked pretty well and it generated the script so I'm not even going to try seeing if it runs because this this
47:43
probably at the bottom here might not run so because this part on the bottom
47:48
might not run I'm going to open up cursor and I'm just going to go to that file i'm going to Yeah so we got to
47:55
comment a few things that Claude did not comment for us before and but now there you go this is
48:02
the file right i just had to do these two commentings and now I'm going to run uvun simple plot and let's see if it
48:09
works so packages already installed everything works beautifully so I didn't
48:14
have to worry about that and let's see if it actually generates the the
48:20
dashboard oh it didn't ah live demos and it's actually not live but yeah all
48:25
right so it didn't work out of the box that's okay here's what I'm going to do i'm going to copy this right and I'm
48:33
going to say uh and I'm going to do what i'm going to
48:40
go to the simple plot file i'm going to just paste the error and the script i'm
48:45
going to copy everything and now that I did that I'm going to paste that to claude so I'm
48:52
going to paste llm there you go uh and
48:57
I'm going to send the same plot right and I'm going to say fix this
49:03
error and follow this again right so
49:09
that's all I'm going to do is just to see if we can actually generate the proper version now because now the model has access to
49:16
the previous version of the script the error that was generated that prompt again and we're saving that to sample
49:23
plot.py so hopefully this time it's going to work pretty well and hopefully
49:28
it will be able to execute what we asked and the core idea here folks that I that
49:33
I'm emphasizing with this is that it's a simple way for you to run generate scripts with a prompt and then run them
49:40
right uv gives you this inline metadata option with the dependencies the Python
49:45
version and by just running UV run in the terminal you get the script running right away which is really cool all
49:51
right so now we can go to sample plot now and we can we can first do a cat
49:58
sample plot okay and now we can go to that file
50:04
sample plot and we can see that the file is there so we can comment out these
50:10
things that should not be here and we can run UV run simple
50:16
plot okay so now there you go so as you can see folks I mean how crazy is that
50:22
so it it plotted some temperature and Bitcoin and crypto stuff and stock price
50:28
information so everything ran out of the box just like that so this is pretty
50:34
awesome and what's cool is that for example uh if I go to one of the files
50:40
that I had like navigate LM output so I have this file for navigating the outputs of an LLM right in the terminal
50:47
but it doesn't have the UV metadata right so I can copy this file and then I
50:53
can go to my clawed projects here where I have the instructions from the UV documentation exactly on how to do that
51:00
inline metadata how it works from the UV documentation and I added those instructions here so I can come here i
51:08
can paste this here in this two that I did with UV with claw projects and hopefully what I would get is there you
51:15
go the creation of that script with the correct UV um uh inline metadata which
51:23
is what you're seeing right here piper clip etc etc with the script etc
51:29
so what I can do is we can copy this and see if it works out of the box so I can
51:35
copy this i'm going to come over here i'm going to paste that to navigate whatever and we're going to run uvun
51:42
navigate and we're going to see if it works the only thing is that I have to run this on some output some input so I
51:49
forgot about that so here's what I'm going to do to fix that
51:54
uh code run so I need a example on how
52:00
to use this so lm uh generate five suggestions for projects involving AI in
52:09
the terminal as bullet points right and now I'm going to pipe this to UV run
52:17
navigate so hopefully this works and there you go so this is really cool
52:23
folks because now this UI this very simple that you're seeing is the Python script running on the output of the LLM
52:30
that I just ran on the terminal so this is just a very simple thing where I can just select whatever the LLM gives me
52:37
and then see what I want to copy to my clipboard and what I don't want to copy so for example uh I want to delete this
52:43
bit delete this this i don't want this maybe like from the examples I don't
52:50
want this one i don't want this one i just want these so now when I'm done I can say copy and exit so I can press C
52:59
yes and now I have all of that copied only the information that I wanted so
53:04
this is the power of integrating everything that we've been talking about with Python scripts to make our
53:10
workflows go completely crazy i mean this is just it gets really cool as you go more and more into uh these ideas so
53:19
we could go on and on but this is pretty much the the core idea that I wanted to discuss now uh I have other examples
53:27
with like highlighting parts and stuff but uh I think this example is good enough and other things that you can do
53:32
is for example you can use the perplexity API to do your own little research assistant right in the terminal and you can integrate MCP servers and a
53:40
bunch of other things now the final example that I want to talk about is Ginga templates which is really really
53:46
nice which is a way to do prompt templates that work directly in your terminal so how do you do that so first
53:53
I'm going to show you how it works so I can for example let's say that um I wanted to do
54:02
um I have one that's for creating Anki flashcards so I'll show you what that means imagine I go to my imagine I go to
54:10
claude right and I get some conversation let's say prompt for thumbnail creation
54:17
uh whisper CPP audio all right so this one okay this one is a pretty good one let me fix edit a fix no this is not
54:25
good enough automating GDPR contract compliance okay
54:30
perfect I think this one is not good enough as well so let's
54:36
see Um let's go to some paper so archive attention is all you need i
54:43
always use this paper for examples so I'm going to copy all this information from this paper all right so I just
54:48
copyr C go to my terminal and I go P and now I'm going to write PMPT which is the
54:54
alias that I have on the terminal and then I'm going to write enky and I'm going to autocomplete with Enki cards
55:00
which is a template that I have written as a Ginga template that creates Enki
55:05
cards that can be automatically uploaded to Enki the the software and then I'm
55:12
going to pipe this to a model so I'm going to pipe this to LM to GPT40
55:18
gpt40 and then I'm going to say I'm not going to say anything so that's all I'm going to take this
55:25
paper pipe it to Anki flashcards to this prompt that uses some sort of way of
55:33
some template to create Anki cards and then I'm going to send that to a model and look what I get in return so I'm
55:40
going to actually save that to Anki cards example no I'm not I'm just going to show you the the output first so
55:49
GPT40 so when I run this folks this is what we get it seems like there's an
55:54
error uh could you provide the actual content all right so let's uh control arr c let's test this out let's see if I
56:02
got the actual contents of the paper yeah I got it so now let's try that
56:07
again yes so now as you can see the cards are being created based on the
56:14
paper and I can if I wanted to automatically upload them to Anki but we're not going to do that right now but
56:20
as you can see the cards are being created they're related to the paper obviously transformer model attention is
56:26
all you need etc etc i could go very specific but now let's see how this type of thing works so what I'm doing here is
56:33
if I go to pmpt and I go which pmpt it indicates that I have a file
56:40
which is a python script called ginga templates ta so first thing is pmpt is a
56:46
little uh it's like a bash function that I have on my um aliases and it it goes
56:53
to this python script so if I copy that and I go cursor this can open up that
57:00
alias hopefully no there you go we can yes so this is the alias and what this
57:07
alias does is essentially I have uh a location for where my template is
57:15
and it's inside of this LM templates folder which is in the um in the current
57:20
folder where this file is located and all this thing is doing is it's a simple function to parse information that's
57:27
coming in from the input from the terminal and another one to render the
57:34
template the ginga template using this basic uh threeline script where I set up
57:40
the environment I load from the templates I get the template name I
57:46
render that template and that's it and this part is just essentially parsing
57:52
the input from the terminal now what's cool is that if I go to Anki card whatever so I'm I'm going to go to the
57:59
correct location i'm going to open up curs on the right folder i'm going to go to I'm going to first this goes away and
58:06
then I'm going to go anky cards ginga.2 this is the template that just went on to create those cards based on the paper
58:13
essentially create this amount of cards for the content below
58:19
and try to capture all the main information and this is the variable content which is the output from the
58:26
clipboard that gets piped to this template so it gets replaced here and
58:31
this other information this is a default uh the default number of cards to create is 10 unless I give it another number
58:39
but usually I just create 10 and that's it the core idea is that I have a bunch
58:46
of ginga chew files to do different things like bash commands bullet instructions enkify full conversation
58:54
enkify a python script uh generate Google search queries this one is actually really cool finally
59:01
there's another cool example which is building command line tools with Rust you can leverage the similar approach that I did with Python scripts but to
59:07
create actual Rust apps that are much faster and more performant to work in
59:13
the terminal with you and I have a few examples and it's you can do that for rewriting tools in the terminal for
59:19
template variation it's you can go wild with this and there's a bunch of fun example use cases that I advise you to
59:25
take a look like creating bulky carts uh integrating tools like wget curl etc uh
59:32
combining commands like mmdc this is a really fun one folks i'm going to show an example this is a command that I ran
59:39
essentially with a single command you generate a full graph in mermaid style
59:44
and render that graph and then you can showcase it so if I go to if I go back
59:50
to where we were before and I run this command yeah with this command what you
59:56
can do is you can generate directly with command an image like this start is it sunny etc etc so
1:00:03
that's pretty cool and there's a bunch of there's so many
1:00:09
things that you can do that go completely wild with this the final example I'm going to show folks is that
1:00:14
you can actually work directly with images to do OCR scan information or ask
1:00:20
an LM what this image is etc for example I can say if I for example take a screenshot
1:00:29
of something for example I'm going to take a screenshot of my of the browser page with my course i can run this
1:00:36
command for example lm extract text scan document and there you go you can extract the text from that image
1:00:42
directly so you can and you can do things like what is this and you can do all sorts of things with images you can
1:00:48
go wild and the APIs are getting cheaper and cheaper so this you can do that for very very cheap at the API call cost and
1:00:57
there's so many other things that you can go on and on and you can work with PDFs using PDFK you can convert
1:01:02
everything to markdown using a tool called markdown there's so many things and the last thing I'll say folks is
1:01:08
that there are layers for working with AI in the terminal just like there are layers for working with AI in general so
1:01:14
you start with basic file creation and editing right so Vim nano etc you create files edit files then you can start
1:01:20
piping stuff like we were showing here to get output from a file send it to LLM
1:01:25
output from your clipboard etc and then you start running commands by doing things like eval this command right for
1:01:33
example this is a way for you to run a bash command directly in the terminal you're moving towards more agency and
1:01:39
more action in your terminal and then you start working with the gent commands like lmter term cloud code a either
1:01:45
magentic 1 cli and so on which are they they go autonomously in your terminal and they start doing stuff for you and
1:01:51
you have this more supervision control over what's going on and then you go fully agentic terminals like warp they
1:01:59
do not uh sponsor this video folks but warp is like an example that they try to do a fully agentic terminal uh that
1:02:06
essentially you ask for things and then things just happen in your terminal pretty cool pretty interesting idea uh
1:02:11
and I like this the set of layers of control because I like actually being
1:02:16
more on these three first rather than in these two but I do like sometimes moving
1:02:23
from let's say this to this when I want a little bit more you know a little bit faster iteration and obviously when I
1:02:30
work in cursor is a merge of having an IDE where I can edit files directly and
1:02:35
cursor has an agentic mode that can create files edit run terminal commands
1:02:41
everything into one so that's why I use it you can also use Windsor for that which is really powerful or you can use
1:02:46
just basic VSCO with GitHub Copilot has similar capabilities and I kind of like
1:02:51
where this evolution of AI is going with especially when working the terminal and that's why I did this video so thanks
1:02:57
for watching folks don't forget to like and subscribe and see you next time

"""

initial_state = {
    "transcription": video_transcription,  # from pyperclip or elsewhere
    "description": "",
    "chapters": "",
    "final_output": ""
}

final_state = graph.invoke(initial_state)

BadRequestError: Error code: 400 - {'error': {'message': "Missing required parameter: 'messages[1].content[0].type'.", 'type': 'invalid_request_error', 'param': 'messages[1].content[0].type', 'code': 'missing_required_parameter'}}